# RLM single-GPU training smoke test
Select a GPU runtime first. This notebook installs the checked-out project, its pinned Colab extra, and the optional Hugging Face dataset adapter; all training logic lives in `rlm_train.colab`.

In [ ]:
%pip install -e . -e './training[colab,hub-datasets]'

## Optional: create deterministic AIME24 and MATH-500 splits
The source revisions and split salts are pinned in `rlm_train.benchmarks`. Artifacts are written to Drive, byte-validated on reruns, and exposed below as ordinary notebook variables. The defaults are project-local AIME24 24/6 and MATH-500 400/100 partitions, not official upstream train/test splits.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DATASET_ROOT = Path('/content/drive/MyDrive/rlm-ib-datasets')

In [ ]:
from rlm_train.benchmarks import prepare_aime24_splits

AIME24_SPLITS = prepare_aime24_splits(DATASET_ROOT / 'aime24')
AIME24_VARIABLES = AIME24_SPLITS.notebook_variables('AIME24')
globals().update(AIME24_VARIABLES)
AIME24_VARIABLES

In [ ]:
from rlm_train.benchmarks import prepare_math500_splits

MATH500_SPLITS = prepare_math500_splits(DATASET_ROOT / 'math500')
MATH500_VARIABLES = MATH500_SPLITS.notebook_variables('MATH500')
globals().update(MATH500_VARIABLES)
MATH500_VARIABLES

In [ ]:
# Optional for an API-judge run; the secret value is never written to config or output.
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
!rlm-train-colab training/configs/colab-smoke.toml

The command above remains the synthetic smoke test; preparing datasets does not silently change a run configuration. For a longer policy baseline, use `training/configs/colab-train.toml` and point a copied dataset/evaluation configuration at the exposed train/test paths. To resume the latest checkpoint, pass `--resume`; an explicit checkpoint directory may follow the flag.